<a href="https://colab.research.google.com/github/mvikman/LLM-course-2025/blob/main/week-2/prompting-notebook/prompting_grumpywizard3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prompting a Large Language Model


This is an example notebook for running an open source model from Hugging Face.

In [2]:
%pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 35.2 MB/s eta 0:00:00


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

We use the [Phi-3.5-mini-instruct](https://huggingface.co/microsoft/Phi-3.5-mini-instruct) in this notebook due to its small size.

In [4]:
torch.random.manual_seed(0)

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3.5-mini-instruct",
    device_map="cuda", # change to cuda if running on GPU
    torch_dtype="auto",
    trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3.5-mini-instruct")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]



```
# This is formatted as code
```

An example few-shot prompt:

In [32]:
messages = [
    {"role": "system", "content": "You are a grumpy and wistful wizard, who likes to use rhymes."},
    {"role": "user", "content": "I require assistance in the field of finance."},
    {"role": "assistant", "content": "Oh sure, ask me your questions of financial lore, I always knew you'd be a terrible bore."},
    {"role": "user", "content": "Investing in wet things, such as Coca Cola or water is safe due to low burn risk, while investing in dry thinks, such as wheat flour or paper, carries a high burn risk. With this in mind, tell me how you would determine safe investment options."},
]

The attempt to introduce the idea of wet things as good investments did not work in the above example. I will here attempt to teach the model this logic with a more clear chain-of-thought prompting.

In [33]:
messages.append({"role": "user", "content": "New logic about safe investments: 1. Dry things burn. 2. Wet things do not burn. 3. Investing in things that burn involves a high risk. Hence it follows that it is safer to invest in things that are wet and cannot burn."})
messages.append({"role": "user", "content": "List some good and safe investment ideas in a table according to their low or high burn risk."})

In [34]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

generation_args = {
    "max_new_tokens": 500,
    "return_full_text": False,
    "do_sample": False,
}

Device set to use cuda


In [35]:
output = pipe(messages, **generation_args)
print(output[0]['generated_text'])

 | Investment Type | Description                                  | Burn Risk Level | Notes                                                                                   |
|-----------------|-----------------------------------------------|-----------------|-----------------------------------------------------------------------------------------|
| Stocks          | Shares in publicly traded companies          | Low             | Diversify across sectors; tech, healthcare, consumer goods often stable                |
| Bonds           | Debt securities issued by governments/corporations | Very Low        | Government bonds (e.g., US Treasury) are considered safe; corporate bonds vary by credit rating |
| Mutual Funds    | Investment programs that pool money to invest in a diversified portfolio | Low to Moderate | Choose funds with a history of stable performance                                      |
| Real Estate     | Property investment (residential, commercial) | Moderate       